# Human-Like Move Picker — **Sync Engine** Training Notebook

This notebook trains a **supervised model** (no RL) that:

1. **Chooses a human-like move** among Stockfish top-20 candidates (XGBoost Ranker)
2. **Predicts think time** for the chosen move (XGBoost Regressor)

✅ Uses a **sync** Stockfish wrapper (no asyncio) so it works smoothly in Jupyter on Windows.

**Requires:**
- `ENGINE_PATH` to your local Stockfish `.exe`
- Prior JSON at `HUMAN_PRIOR_PATH` (your 2200–2400 band)
- Azure SQL creds for `dbo.game_core` + `dbo.game_text`


## 0) Install deps (once per kernel)

In [1]:
%pip -q install xgboost pandas pyarrow scikit-learn python-dotenv python-chess pyodbc stockfish
print('✅ Dependencies installed')

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed


## 1) Set project root + import helpers

In [2]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

# 👇 CHANGE this if your repo is elsewhere
PROJECT_ROOT = Path(r"E:\Projects\AIP\DeepChessIQ")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure packages are packages
for p in [
    PROJECT_ROOT/"chess_bot"/"__init__.py",
    PROJECT_ROOT/"chess_bot"/"engine"/"__init__.py",
    PROJECT_ROOT/"chess_bot"/"policy"/"__init__.py",
    PROJECT_ROOT/"chess_bot"/"data"/"__init__.py",
]:
    p.parent.mkdir(parents=True, exist_ok=True)
    p.touch(exist_ok=True)

load_dotenv()
print('CWD =', os.getcwd())
print('ENGINE_PATH =', os.getenv('ENGINE_PATH'))
print('HUMAN_PRIOR_PATH =', os.getenv('HUMAN_PRIOR_PATH'))

CWD = E:\Projects\AIP\DeepChessIQ
ENGINE_PATH = None
HUMAN_PRIOR_PATH = None


## 2) Define **sync** Stockfish service (no asyncio) + sanity check

In [3]:
from pathlib import Path
from stockfish import Stockfish

ENGINE_PATH = os.getenv("ENGINE_PATH")  # set below if None
ENGINE_THREADS = int(os.getenv("ENGINE_THREADS", "2"))
ENGINE_HASH_MB = int(os.getenv("ENGINE_HASH_MB", "128"))
SHORTLIST_N = int(os.getenv("SHORTLIST_N", "20"))

class StockfishServiceSync:
    def __init__(self, path: str = None):
        self.path = path or ENGINE_PATH
        self.sf = None
    def open(self):
        if not self.path or not Path(self.path).exists():
            raise FileNotFoundError(f"ENGINE_PATH invalid: {self.path!r}")
        self.sf = Stockfish(path=self.path, parameters={
            "Threads": ENGINE_THREADS,
            "Hash": ENGINE_HASH_MB,
            "MultiPV": max(1, SHORTLIST_N),
        })
    def close(self):
        self.sf = None
    def get_top_moves(self, fen: str, n: int = SHORTLIST_N):
        self.sf.set_fen_position(fen)
        self.sf.update_engine_parameters({"MultiPV": max(1, n)})
        top = self.sf.get_top_moves(n)
        out = []
        for m in top or []:
            out.append({
                "uci": m.get("Move"),
                "score_cp": m.get("Centipawn"),
                "mate": m.get("Mate"),
                "depth": None,
                "pv": None,
            })
        return {"top_moves": out}

from chess_bot.policy.human_prior_store import HumanPriorStore

# 🔧 If ENGINE_PATH isn't set in .env, set it here once
if not ENGINE_PATH:
    os.environ["ENGINE_PATH"] = r"E:\\Projects\\AIP\\stockfish\\stockfish-windows-x86-64-avx2.exe"
    ENGINE_PATH = os.getenv("ENGINE_PATH")

print('ENGINE_PATH exists =', Path(ENGINE_PATH).exists())
svc = StockfishServiceSync(); svc.open()
fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
res = svc.get_top_moves(fen, n=20)
print(len(res["top_moves"]), 'candidates; top1 =', res["top_moves"][0]["uci"] if res["top_moves"] else None)
svc.close()
store = HumanPriorStore(); print('Prior store OK')

ENGINE_PATH exists = True
20 candidates; top1 = d2d4
Prior store OK


## 3) Candidate builder (engine injected)

In [4]:
# --- Hardcoded SQL settings (training only) ---
SQL_SERVER   = "tcp:64squares.database.windows.net,1433"  # tcp + port avoids Named Pipes
SQL_DB       = "64Squares"
SQL_USER     = "Squares"
SQL_PASSWORD = "Chess@123"
SQL_DRIVER   = "ODBC Driver 18 for SQL Server"

import pyodbc
cs = (
    f"DRIVER={{{SQL_DRIVER}}};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DB};"
    f"UID={SQL_USER};"
    f"PWD={SQL_PASSWORD};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)
# Smoke test: open + close
_conn = pyodbc.connect(cs)
print("✅ Connected to", SQL_SERVER, "/", SQL_DB)
_conn.close()


✅ Connected to tcp:64squares.database.windows.net,1433 / 64Squares


In [5]:
# Re-define build_candidates_with_engine() to use hardcoded creds above
import io, re, time
import numpy as np, pandas as pd, pyodbc
import chess, chess.pgn
from pathlib import Path

TC_RE  = re.compile(r"^\s*(\d+)(?:\+(\d+))?\s*$")
CLK_RE = re.compile(r"\[\s*%clk\s+([0-9:]+)\s*\]")

def clock_to_ms(clk: str):
    if not clk: return None
    parts = [int(p) for p in clk.split(":")]
    if   len(parts)==2: h,m,s = 0, parts[0], parts[1]
    elif len(parts)==3: h,m,s = parts
    else: return None
    return ((h*60+m)*60+s)*1000

def parse_timecontrol(tc: str):
    if not tc: return (None, None)
    m = TC_RE.match(tc.strip())
    if not m: return (None, None)
    return int(m.group(1))*1000, int(m.group(2) or 0)*1000

def pgntxt(start_fen, movetext):
    headers = ['[Event "-" ]','[Site "-" ]','[Date "????.??.??"]','[Round "-" ]',
               '[White "-" ]','[Black "-" ]','[Result "*" ]','[SetUp "1"]', f'[FEN "{start_fen}"]']
    body = movetext.strip()
    if not body.endswith(("1-0","0-1","1/2-1/2","*")): body += " *"
    return "\n".join(headers) + "\n\n" + body + "\n"

def iter_plies(start_fen, movetext_full, timecontrol):
    base_ms, inc_ms = parse_timecontrol(timecontrol or "")
    game = chess.pgn.read_game(io.StringIO(pgntxt(start_fen, movetext_full)))
    if not game: return
    board = game.board()
    prev_post = {True: None, False: None}
    ply = 0
    for node in game.mainline():
        if node.move is None: continue
        fen_before = board.fen()
        side = board.turn
        uci = node.move.uci()
        san = board.san(node.move)
        board.push(node.move)
        ply += 1
        post_ms = None
        if node.comment:
            m = CLK_RE.search(node.comment)
            if m: post_ms = clock_to_ms(m.group(1))
        think_ms = None
        if post_ms is not None and base_ms is not None:
            pre = base_ms if prev_post[side] is None else max(prev_post[side] + (inc_ms or 0), 0)
            d = pre - post_ms
            if 0 <= d <= 10*60*1000: think_ms = d
        prev_post[side] = post_ms
        yield {"ply": ply, "fen": fen_before, "side": "w" if side else "b",
               "human_uci": uci, "human_san": san, "think_ms": think_ms}

def build_candidates_with_engine(
    svc, store, *,
    elo_min=2200, elo_max=2400,
    max_positions=1000, keep_max_ply=60,
    print_every=200,
    out_parquet="data/candidates_2200_2400.parquet",
    shortlist_n=20
):
    # Use the hardcoded connection string from cell #1
    cs = (
        f"DRIVER={{{SQL_DRIVER}}};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DB};"
        f"UID={SQL_USER};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
    )
    print(f"[SQL] Connecting to {SQL_SERVER}/{SQL_DB} …")
    conn = pyodbc.connect(cs)
    cur = conn.cursor()
    print("[SQL] Querying games …")
    cur.execute(f"""
        SELECT TOP {max_positions*2}
          core.game_pk,
          COALESCE(NULLIF(core.start_fen,''),'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1') AS start_fen,
          COALESCE(core.timecontrol,'') AS timecontrol,
          txt.pgn_movetext_full
        FROM dbo.game_core core
        JOIN dbo.game_text txt ON txt.game_pk = core.game_pk
        WHERE txt.pgn_movetext_full IS NOT NULL
          AND core.elo_avg BETWEEN ? AND ?
        ORDER BY core.game_pk;
    """, (elo_min, elo_max))
    rows = cur.fetchall()
    print(f"[SQL] Retrieved {len(rows)} games.")

    import numpy as np, pandas as pd, time
    recs, kept_positions = [], 0
    t0 = time.time()

    for ridx, r in enumerate(rows, 1):
        if ridx % max(1, print_every//10) == 0:
            print(f"[SCAN] Game #{ridx} (kept={kept_positions}) …")
        try:
            for pl in iter_plies(r.start_fen, r.pgn_movetext_full, r.timecontrol):
                if pl["ply"] > keep_max_ply: break
                fen, side, human_uci, think_ms = pl["fen"], pl["side"], pl["human_uci"], pl["think_ms"]
                eng = svc.get_top_moves(fen, n=shortlist_n)["top_moves"]
                if not eng: continue
                uci_list = [m["uci"] for m in eng]
                if human_uci not in uci_list:  # only keep where human is within engine shortlist
                    continue

                prior = store.get_prior(fen)
                freq_map = {m["uci"]: m["freq"] for m in prior.get("moves", [])}
                mean_map = {m["uci"]: m.get("mean_ms") for m in prior.get("moves", [])}
                total_freq = prior.get("total", 0) or 1

                cp_vals = [(-10**9 if m.get("score_cp") is None else m["score_cp"]) for m in eng]
                cp_arr = np.array(cp_vals, dtype=np.float32)
                tau = 60.0
                exps = np.exp((cp_arr/tau) - (np.max(cp_arr)/tau))
                p_engine = (exps / (exps.sum() or 1.0)).tolist()
                best_cp = max([m["score_cp"] for m in eng if m.get("score_cp") is not None], default=None)

                group_id = f"{hash(fen)}:{kept_positions}"
                for i, m in enumerate(eng):
                    uci = m["uci"]
                    recs.append({
                        "group_id": group_id,
                        "fen": fen,
                        "side": 1 if side=="w" else 0,
                        "uci": uci,
                        "label_choice": 1 if uci==human_uci else 0,
                        "cp": m.get("score_cp") or 0,
                        "mate": m.get("mate") or 0,
                        "depth": m.get("depth") or 0,
                        "pv_len": len(m.get("pv","").split()) if m.get("pv") else 0,
                        "engine_soft_p": p_engine[i],
                        "prior_freq": freq_map.get(uci, 0),
                        "prior_prob": (freq_map.get(uci, 0)/total_freq),
                        "prior_mean_ms": (mean_map.get(uci) or 0),
                        "human_think_ms": think_ms,
                        "ply": pl["ply"],
                        "legal_count": len(uci_list),
                        "cp_to_best": (0 if (best_cp is None or m.get("score_cp") is None) else best_cp - m["score_cp"]),
                    })
                kept_positions += 1
                if kept_positions % print_every == 0:
                    print(f"[KEEP] kept={kept_positions} rows={len(recs)}")
                if kept_positions >= max_positions:
                    raise StopIteration
        except StopIteration:
            break

    df = pd.DataFrame.from_records(recs)
    Path(out_parquet).parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_parquet, index=False)
    dt = time.time() - t0
    print(f"[OK] wrote {len(df)} rows to {out_parquet} in {dt:.1f}s (≈{len(df)/max(1,kept_positions):.1f}/pos)")
    cur.close(); conn.close()
    return df


In [6]:
def _sql_fetch_games(elo_min, elo_max, limit_rows):
    cs = (
        f"DRIVER={{{SQL_DRIVER}}};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DB};"
        f"UID={SQL_USER};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
    )
    conn = pyodbc.connect(cs)
    cur = conn.cursor()
    cur.execute(f"""
        SELECT TOP {limit_rows}
          core.game_pk,
          COALESCE(NULLIF(core.start_fen,''),'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1') AS start_fen,
          COALESCE(core.timecontrol,'') AS timecontrol,
          txt.pgn_movetext_full
        FROM dbo.game_core core
        JOIN dbo.game_text txt ON txt.game_pk = core.game_pk
        WHERE txt.pgn_movetext_full IS NOT NULL
          AND core.elo_avg BETWEEN ? AND ?
        ORDER BY core.game_pk;
    """, (elo_min, elo_max))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return rows


### 3a) Build a small candidate set to verify

In [7]:
# ---- FEN cache + adaptive ladder helpers ----
from collections import OrderedDict
import numpy as np

class FenCache:
    def __init__(self, max_items=200_000):
        self.max = max_items
        self._d = OrderedDict()
    def get(self, k):
        v = self._d.get(k)
        if v is not None:
            self._d.move_to_end(k)
        return v
    def put(self, k, v):
        self._d[k] = v
        self._d.move_to_end(k)
        if len(self._d) > self.max:
            self._d.popitem(last=False)

FEN_CACHE = FenCache(max_items=200_000)

def get_top_moves_cached(svc, fen, n=20):
    key = (fen, n)
    hit = FEN_CACHE.get(key)
    if hit is not None:
        return hit
    res = svc.get_top_moves(fen, n=n)
    FEN_CACHE.put(key, res)
    return res

def prior_top_share(prior_dict):
    total = max(1, prior_dict.get("total", 0))
    mv = prior_dict.get("moves") or []
    if not mv:
        return 0.0
    top = max(mv, key=lambda m: m.get("freq", 0))
    return (top.get("freq", 0) / total)

def get_top_moves_until_contains(svc, fen, target_uci, *, steps=(6,12,20)):
    """
    Ask for top-k in an increasing ladder until the human move appears,
    reusing results via cache to minimize engine calls.
    """
    last = None
    for k in steps:
        last = get_top_moves_cached(svc, fen, n=k)["top_moves"]
        if not last:
            return None
        if target_uci in [m.get("uci") for m in last]:
            return last
    return last  # may or may not include target; caller decides


In [8]:
print(SQL_DRIVER)

ODBC Driver 18 for SQL Server


In [9]:
# ---- PARALLEL BUILDER (optional) with ladder + coverage ----
import os, time, math, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd, pyodbc
from pathlib import Path

def _sql_fetch_games(elo_min, elo_max, limit_rows):
    cs = (
        f"DRIVER={{{SQL_DRIVER}}};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DB};"
        f"UID={SQL_USER};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
    )
    conn = pyodbc.connect(cs)
    cur = conn.cursor()
    cur.execute(f"""
        SELECT TOP {limit_rows}
          core.game_pk,
          COALESCE(NULLIF(core.start_fen,''),'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1') AS start_fen,
          COALESCE(core.timecontrol,'') AS timecontrol,
          txt.pgn_movetext_full
        FROM dbo.game_core core
        JOIN dbo.game_text txt ON txt.game_pk = core.game_pk
        WHERE txt.pgn_movetext_full IS NOT NULL
          AND core.elo_avg BETWEEN ? AND ?
        ORDER BY core.game_pk;
    """, (elo_min, elo_max))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return rows

def _process_batch(rows_batch, shortlist_n, keep_max_ply, prior_share_threshold, steps_train):
    from chess_bot.policy.human_prior_store import HumanPriorStore
    svc_local = StockfishServiceSync(); svc_local.open()
    store_local = HumanPriorStore()

    recs = []
    kept_positions = 0
    seen_positions = 0
    hits_positions = 0

    for r in rows_batch:
        for pl in iter_plies(r.start_fen, r.pgn_movetext_full, r.timecontrol):
            if pl["ply"] > keep_max_ply: break
            fen, side, human_uci, think_ms = pl["fen"], pl["side"], pl["human_uci"], pl["think_ms"]

            prior = store_local.get_prior(fen)
            share = prior_top_share(prior)
            ladder = (6,12) if share >= prior_share_threshold else steps_train

            seen_positions += 1
            eng = get_top_moves_until_contains(svc_local, fen, human_uci, steps=ladder)
            if not eng: 
                continue
            uci_list = [m["uci"] for m in eng]
            if human_uci not in uci_list:
                continue
            hits_positions += 1

            freq_map = {m["uci"]: m["freq"] for m in prior.get("moves", [])}
            mean_map = {m["uci"]: m.get("mean_ms") for m in prior.get("moves", [])}
            total_freq = prior.get("total", 0) or 1

            cp_vals = [(-10**9 if m.get("score_cp") is None else m["score_cp"]) for m in eng]
            cp_arr = np.array(cp_vals, dtype=np.float32)
            tau = 60.0
            exps = np.exp((cp_arr/tau) - (np.max(cp_arr)/tau))
            p_engine = (exps / (exps.sum() or 1.0)).tolist()
            best_cp = max([m["score_cp"] for m in eng if m.get("score_cp") is not None], default=None)

            group_id = f"{hash(fen)}:{threading.get_ident()}:{kept_positions}"
            for i, m in enumerate(eng):
                uci = m["uci"]
                recs.append({
                    "group_id": group_id,
                    "fen": fen,
                    "side": 1 if side=="w" else 0,
                    "uci": uci,
                    "label_choice": 1 if uci==human_uci else 0,
                    "cp": m.get("score_cp") or 0,
                    "mate": m.get("mate") or 0,
                    "depth": m.get("depth") or 0,
                    "pv_len": len(m.get("pv","").split()) if m.get("pv") else 0,
                    "engine_soft_p": p_engine[i],
                    "prior_freq": freq_map.get(uci, 0),
                    "prior_prob": (freq_map.get(uci, 0)/total_freq),
                    "prior_mean_ms": (mean_map.get(uci) or 0),
                    "human_think_ms": think_ms,
                    "ply": pl["ply"],
                    "legal_count": len(uci_list),
                    "cp_to_best": (0 if (best_cp is None or m.get("score_cp") is None) else best_cp - m["score_cp"]),
                })
            kept_positions += 1

    try: svc_local.close()
    except: pass
    return recs, seen_positions, hits_positions


# Patch: finer-grained batching for smoother tqdm progress
def build_candidates_parallel(*,
    elo_min=2200, elo_max=2400,
    max_positions=1000, keep_max_ply=60,
    shortlist_n=20,
    workers=None, out_parquet="data/candidates_2200_2400.parquet",
    sql_fetch_multiplier=2,
    steps_train=(6,12,20,32),
    prior_share_threshold=0.60,
    num_batches=None,                 # NEW: total chunks to split into
):
    import os, math, time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm.auto import tqdm
    t0 = time.time()

    rows = _sql_fetch_games(elo_min, elo_max, limit_rows=max_positions*sql_fetch_multiplier)
    if not rows:
        raise RuntimeError("No rows returned from SQL")

    workers = workers or max(2, min(4, (os.cpu_count() or 4)//2))
    # default to finer granularity so the bar moves often
    num_batches = num_batches or (workers * 8)
    num_batches = max(num_batches, workers)  # at least one batch per worker

    chunk_size = max(1, math.ceil(len(rows) / num_batches))
    chunks = [rows[i:i+chunk_size] for i in range(0, len(rows), chunk_size)]

    recs_all = []
    kept_positions = 0
    seen_all, hits_all = 0, 0

    with ThreadPoolExecutor(max_workers=workers) as ex, \
         tqdm(total=len(chunks), desc=f"Batches (workers={workers})", unit="chunk") as tq:
        futs = [ex.submit(_process_batch, ch, shortlist_n, keep_max_ply, prior_share_threshold, steps_train)
                for ch in chunks]
        for fut in as_completed(futs):
            recs, seen, hits = fut.result()
            recs_all.extend(recs)
            seen_all += seen
            hits_all += hits
            kept_positions += len({r["group_id"] for r in recs})

            cov = (hits_all / max(1, seen_all)) * 100.0
            tq.update(1)
            tq.set_postfix(rows=len(recs_all), kept=kept_positions, cov=f"{cov:.1f}%")

            if kept_positions >= max_positions:
                tq.set_postfix_str("Target positions reached; stopping early.")
                break

    import pandas as pd
    from pathlib import Path
    df = pd.DataFrame.from_records(recs_all)
    if len(df):
        keep_groups = list(df["group_id"].drop_duplicates().head(max_positions))
        df = df[df["group_id"].isin(keep_groups)].reset_index(drop=True)

    Path(out_parquet).parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_parquet, index=False)
    dt = time.time() - t0
    cov_final = (hits_all / max(1, seen_all)) * 100.0
    print(f"[OK] wrote {len(df)} rows to {out_parquet} in {dt:.1f}s (groups={df['group_id'].nunique() if len(df) else 0})")
    print(f"[COVERAGE] human-in-shortlist: {hits_all}/{seen_all} = {cov_final:.1f}%")
    return df


In [10]:
from tqdm.auto import tqdm

c:\Users\ppava\anaconda3\envs\chess-bot-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = build_candidates_parallel(
    elo_min=2200, elo_max=2400,
    max_positions=500,
    keep_max_ply=60,
    shortlist_n=20,
    workers=6,
    steps_train=(6,12,20),
    prior_share_threshold=0.60,
    num_batches=96,   # <= try 32–64 for nice, frequent updates
    out_parquet="data/candidates_2200_2400.parquet"
)

Batches (workers=6):   0%|          | 0/91 [00:00<?, ?chunk/s]

## 4) Train XGBoost Ranker + Time Regressor

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

DATA_PATH = os.getenv('CANDIDATE_OUT', 'data/candidates_2200_2400.parquet')
MODEL_DIR = Path(os.getenv('MODEL_OUT', 'models/human_pick'))
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
grp_has_pos = df.groupby('group_id')['label_choice'].max()
keep_groups = grp_has_pos[grp_has_pos > 0].index
df = df[df['group_id'].isin(keep_groups)].reset_index(drop=True)
print('Rows after enforcing human-in-top20:', len(df))

feat_cols = [
    'side','cp','mate','depth','pv_len','engine_soft_p',
    'prior_freq','prior_prob','prior_mean_ms',
    'ply','legal_count','cp_to_best'
]
for c in feat_cols:
    df[c] = df[c].fillna(0)

groups = df['group_id'].unique()
train_g, val_g = train_test_split(groups, test_size=0.15, random_state=42)
train = df[df.group_id.isin(train_g)].copy()
val   = df[df.group_id.isin(val_g)].copy()

def make_dm(d):
    X = d[feat_cols].astype(np.float32).values
    y = d['label_choice'].astype(np.float32).values
    gsizes = d.groupby('group_id')['uci'].count().to_list()
    dm = xgb.DMatrix(X, label=y)
    dm.set_group(gsizes)
    return dm

dtrain = make_dm(train); dval = make_dm(val)
params = dict(objective='rank:pairwise', eval_metric='ndcg@1', tree_method='hist',
              max_depth=6, eta=0.08, subsample=0.9, colsample_bytree=0.9, min_child_weight=10)

rk = xgb.train(params=params, dtrain=dtrain, num_boost_round=800,
               evals=[(dtrain,'train'),(dval,'val')], early_stopping_rounds=50, verbose_eval=50)
rk.save_model(str(MODEL_DIR / 'xgb_ranker.json'))

val['util'] = rk.predict(xgb.DMatrix(val[feat_cols].astype(np.float32).values))
pred_idx = val.groupby('group_id')['util'].idxmax(); pred = val.loc[pred_idx]
acc_top1 = pred['label_choice'].mean()
print(f"Top-1 human match (val): {acc_top1:.3f}")

time_rows = df[(df['label_choice']==1) & df['human_think_ms'].notna()].copy()
if len(time_rows) > 1000:
    time_rows['log_time'] = np.log1p(time_rows['human_think_ms'].clip(lower=0))
    time_feats = feat_cols
    Xtr, Xte, ytr, yte = train_test_split(time_rows[time_feats].astype(np.float32).values,
                                          time_rows['log_time'].values, test_size=0.15, random_state=42)
    dtr = xgb.DMatrix(Xtr, label=ytr); dte = xgb.DMatrix(Xte, label=yte)
    rt = xgb.train(params=dict(objective='reg:squarederror', eval_metric='rmse', max_depth=6, eta=0.08,
                               tree_method='hist', subsample=0.9, colsample_bytree=0.9, min_child_weight=10),
                   dtrain=dtr, num_boost_round=600, evals=[(dtr,'train'),(dte,'val')],
                   early_stopping_rounds=50, verbose_eval=50)
    rt.save_model(str(MODEL_DIR / 'xgb_time.json'))
    pred_log = rt.predict(dte)
    mae_s = mean_absolute_error(np.expm1(yte)/1000.0, np.expm1(pred_log)/1000.0)
    print(f"Time MAE (seconds): {mae_s:.2f}")
    json.dump({'feat_cols': feat_cols, 'time_feat_cols': time_feats}, open(MODEL_DIR/'meta.json','w'))
else:
    print('Not enough rows with think_ms to train time regressor; saving ranker only.')
    json.dump({'feat_cols': feat_cols, 'time_feat_cols': None}, open(MODEL_DIR/'meta.json','w'))

print('✅ Models saved to', MODEL_DIR)

## 5) Inference — pick human-like move + predicted time

In [ ]:
import numpy as np, xgboost as xgb, chess
from pathlib import Path

MODEL_DIR = Path(os.getenv('MODEL_OUT', 'models/human_pick'))
meta = json.load(open(MODEL_DIR/'meta.json'))
FEATS = meta['feat_cols']; TIME_FEATS = meta.get('time_feat_cols')

rk = xgb.Booster(); rk.load_model(str(MODEL_DIR/'xgb_ranker.json'))
rt = None
if TIME_FEATS and (MODEL_DIR/'xgb_time.json').exists():
    rt = xgb.Booster(); rt.load_model(str(MODEL_DIR/'xgb_time.json'))

svc = StockfishServiceSync(); svc.open()
store = HumanPriorStore()

def build_rows_for_fen(fen: str):
    eng = svc.get_top_moves(fen, n=int(os.getenv('SHORTLIST_N','20')))['top_moves']
    if not eng: return [], []
    uci_list = [m['uci'] for m in eng]
    best_cp = max([m['score_cp'] for m in eng if m.get('score_cp') is not None], default=None)
    cp_list = [(-10**9 if m.get('score_cp') is None else m['score_cp']) for m in eng]
    tau = 60.0
    exps = np.exp((np.array(cp_list)/tau) - (np.max(np.array(cp_list))/tau))
    pe = (exps / (exps.sum() or 1.0)).tolist()
    prior = store.get_prior(fen)
    freq_map = {m['uci']: m['freq'] for m in prior.get('moves', [])}
    mean_map = {m['uci']: m.get('mean_ms') for m in prior.get('moves', [])}
    total = prior.get('total', 0) or 1
    prior_probs = [(freq_map.get(u,0)/total) for u in uci_list]
    board = chess.Board(fen); side = 1 if board.turn else 0
    legal_count = len(uci_list); ply = board.fullmove_number*2 - (0 if board.turn else 1)
    rows = []
    for i, m in enumerate(eng):
        rows.append(dict(
            side=side, cp=(m.get('score_cp') or 0), mate=(m.get('mate') or 0),
            depth=(m.get('depth') or 0), pv_len=len(m.get('pv','').split()) if m.get('pv') else 0,
            engine_soft_p=pe[i], prior_freq=freq_map.get(m['uci'], 0), prior_prob=prior_probs[i],
            prior_mean_ms=(mean_map.get(m['uci']) or 0), ply=ply, legal_count=legal_count,
            cp_to_best=(0 if (best_cp is None or m.get('score_cp') is None) else best_cp - m['score_cp']),
        ))
    return rows, uci_list

def pick_move(fen: str):
    rows, uci_list = build_rows_for_fen(fen)
    if not rows:
        return None
    X = np.array([[r[c] for c in FEATS] for r in rows], dtype=np.float32)
    util = rk.predict(xgb.DMatrix(X))
    idx = int(np.argmax(util))
    picked = uci_list[idx]
    think_ms = None
    if rt is not None and TIME_FEATS:
        x_time = np.array([[rows[idx][c] for c in TIME_FEATS]], dtype=np.float32)
        log_ms = rt.predict(xgb.DMatrix(x_time))[0]
        think_ms = int(np.expm1(log_ms))
    return dict(fen=fen, picked=picked, util=float(util[idx]), pred_think_ms=think_ms)

fen = "r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 2 4"
out = pick_move(fen)
svc.close()
out

## 6) Next steps
- Increase `CANDIDATE_MAX_POS` once metrics look sane.
- Add more features (captures, checks, promotions) to boost accuracy.
- Add a pre-check at inference: take +mate, avoid −mate.